# Plot: Judge vs Fan Attention Heatmap (Model III)
Extracted from task3/dwts/analysis_task3.py


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

PAPER_RC = {
    # Font sizes
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 9,
    "figure.titlesize": 12,

    # Figure settings
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "figure.figsize": (6.5, 3.5),

    # Style
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "axes.axisbelow": True,

    # Font family
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
}

def apply_paper_style():
    """Apply paper-ready matplotlib style."""
    plt.rcParams.update(PAPER_RC)

apply_paper_style()


In [ ]:
# =========================
# 1) 路径与配置
# =========================
# 默认从 notebooks/26Feb2-mcm 目录运行
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
DATA_DIR = os.path.join(BASE_DIR, "data", "26Feb2-mcm")
FIGDIR = os.path.join(BASE_DIR, "figures", "26Feb2-mcm", "13_judge_fan_heatmap")

# LaTeX single-column figure sizes
FIGSIZE_STANDARD = (6.5, 3.5)
FIGSIZE_WIDE = (6.5, 2.8)
FIGSIZE_TALL = (6.5, 4.5)
FIGSIZE_DUAL = (8.6, 3.2)
DPI = 300

FIGSIZE = FIGSIZE_DUAL

JUDGE_CSV = os.path.join(DATA_DIR, "13_judge_fan_heatmap", "head_attention_judge.csv")
FAN_CSV = os.path.join(DATA_DIR, "13_judge_fan_heatmap", "head_attention_fan.csv")
OUT_PDF = os.path.join(FIGDIR, "judge_vs_fan_contrib_heatmap.pdf")

os.makedirs(FIGDIR, exist_ok=True)


In [ ]:
# Paper style settings
CMA_ATT = "YlGnBu"

DISPLAY_NAMES = {
    "ballroom_partner": "Partner",
    "celebrity_industry": "Industry",
    "celebrity_homestate": "Home State",
    "celebrity_homecountry/region": "Country",
    "age_z": "Age",
    "partner_score_z": "Partner Skill",
    "progress_z": "Progress",
    "popularity_z": "Popularity",
}

def plot_judge_fan_heatmap(judge_path: str, fan_path: str, output_path: str):
    """Generate judge vs fan attention heatmap."""
    head_j = pd.read_csv(judge_path)
    head_f = pd.read_csv(fan_path)

    token_order = [c for c in head_j.columns if c != "head"]
    x_labels = [DISPLAY_NAMES.get(t, t) for t in token_order]

    judge_data = head_j[token_order].values
    fan_data = head_f[token_order].values

    vmax = float(max(judge_data.max(), fan_data.max(), 1e-9))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIGSIZE)

    im1 = ax1.imshow(judge_data, vmin=0, vmax=vmax, cmap=CMA_ATT, aspect="auto")
    im2 = ax2.imshow(fan_data, vmin=0, vmax=vmax, cmap=CMA_ATT, aspect="auto")

    for ax, title in [(ax1, "Judges"), (ax2, "Fans")]:
        ax.set_xticks(np.arange(len(token_order)))
        ax.set_xticklabels(x_labels, rotation=25, ha="right")
        ax.set_yticks(np.arange(judge_data.shape[0]))
        ax.set_yticklabels([f"Head {i}" for i in range(judge_data.shape[0])])
        ax.set_title(title)

    fig.colorbar(im2, ax=[ax1, ax2], fraction=0.03, pad=0.02)
    fig.suptitle("Judge vs Fan: Attention Allocation over Attributes", x=0.48, y=0.97, ha="center", fontsize=12, fontweight="bold")

    plt.tight_layout()
    plt.savefig(output_path, dpi=DPI, bbox_inches="tight")
    plt.close()
    print(f"Saved: {output_path}")


In [ ]:
plot_judge_fan_heatmap(JUDGE_CSV, FAN_CSV, OUT_PDF)
print("Saved:", OUT_PDF)
